# Live demo: SKG-IF APIs for ORKG and Wikidata

[HERITRACE](https://github.com/opencitations/heritrace) is a software for semantic data editing, presented in a [journal article](https://doi.org/10.6092/issn.2532-8816/21218).
The article metadata is available on [ORKG](https://orkg.org/), while the software metadata is available on [Wikidata](https://www.wikidata.org/).

This notebook demonstrates how RAMOSE retrieves both types of information from different data sources and returns them in a single, [SKG-IF](https://skg-if.github.io/interoperability-framework/) compatible schema.

To run it interactively, click **Open in Colab** in the top bar.

In [ ]:
!pip install -q -U ramose

In [ ]:
from importlib.metadata import version

print("ramose", version("ramose"))

In [ ]:
from urllib.request import urlretrieve

urlretrieve(
    "https://raw.githubusercontent.com/opencitations/ramose/master/docs/orkg_skgif.hf",
    "orkg_skgif.hf",
)
urlretrieve(
    "https://raw.githubusercontent.com/opencitations/ramose/master/docs/wikidata_skgif.hf",
    "wikidata_skgif.hf",
)

## Query HERITRACE as a journal article (ORKG)

In [ ]:
import json

from ramose import APIManager

api_manager = APIManager(["orkg_skgif.hf"])
op = api_manager.get_op("/skgif-orkg/v1/products/http://orkg.org/orkg/resource/R1637585")
status, body, content_type, headers = op.exec()
print(json.dumps(json.loads(body), indent=2))

## Query HERITRACE as a software (Wikidata)

In [ ]:
wikidata_manager = APIManager(["wikidata_skgif.hf"])
op = wikidata_manager.get_op("/skgif-wikidata/v1/products/http://www.wikidata.org/entity/Q139571902")
status, body, content_type, headers = op.exec()
print(json.dumps(json.loads(body), indent=2))

## Generated HTML documentation

RAMOSE also generates interactive API documentation from the same spec file. When RAMOSE runs as a web server (`ramose -s spec.hf -w 127.0.0.1:8080`), it serves this page at each API's base path (for example `/skgif-orkg/v1`).

In [ ]:
import base64

from IPython.display import IFrame

from ramose import HTMLDocumentationHandler

doc_handler = HTMLDocumentationHandler(api_manager)
_, html_content = doc_handler.get_documentation()

IFrame(
    src="data:text/html;base64," + base64.b64encode(html_content.encode()).decode(),
    width="100%",
    height=600,
)

## Generated OpenAPI specification

RAMOSE generates an [OpenAPI 3.1](https://spec.openapis.org/oas/v3.1.0) specification from the same spec file. When RAMOSE runs as a web server (`ramose -s spec.hf -w 127.0.0.1:8080`), it serves the spec at each API's `openapi.yaml` (for example `/skgif-orkg/v1/openapi.yaml`).

In [ ]:
from ramose import OpenAPIDocumentationHandler

openapi_handler = OpenAPIDocumentationHandler(api_manager)
_, openapi_yaml = openapi_handler.get_documentation()
print(openapi_yaml)

## Interactive Swagger UI

The same specification renders as interactive [Swagger UI](https://swagger.io/tools/swagger-ui/). When RAMOSE runs as a web server (`ramose -s spec.hf -w 127.0.0.1:8080`), it serves Swagger UI at `/docs` with assets bundled (no CDN). Here `get_swagger_ui` returns a page that loads Swagger UI from a CDN and inlines the spec, embedded in a sandboxed iframe.

In [ ]:
import html

from IPython.display import HTML

_, swagger_html = openapi_handler.get_swagger_ui()

HTML(
    '<iframe sandbox="allow-scripts allow-same-origin" style="width:100%;height:600px;border:0" '
    'srcdoc="' + html.escape(swagger_html, quote=True) + '"></iframe>'
)